# QFF 2026 Phase 2 — PS-O1

## Quantum Optimization of Network Partitioning Using QAOA

**Team:** Quantum Drifters  
**Problem:** weighted Max-Cut on the organizer-locked six-vertex instance  
**Reproducibility:** official organizer utilities, fixed seeds, 4096 shots per sampled result.

Run every cell from top to bottom. The organizer module `qff_phase2.py` is used unchanged. Processor A is explicitly reported as non-executable because it has only five qubits.

## 0. Imports, official settings, and output folders

All required benchmark settings come directly from `qff_phase2.py`; the notebook does not override them. The five optimization starts use `numpy.random.default_rng(seed)` for seeds 0 through 4.

In [ ]:
from pathlib import Path
import hashlib
import itertools
import json
import math
import platform

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.optimize import brute, minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector

from qff_phase2 import (
    BASIS_GATES, OPT_LEVEL, SEED, SHOTS, PROCESSOR_A, PROCESSOR_B,
    bell_experiment, circuit_metrics, counts_to_x, load_processor,
    simulator, transpile_for, validate_processor_d,
)

SEED_TRANSPILER = 42
SEED_SIMULATOR = 42
INITIALIZATION_SEEDS = [0, 1, 2, 3, 4]
OPTIMIZER_NAME = "L-BFGS-B"
OPTIMIZER_MAXITER = 500
GRID_POINTS = 101
OUTPUT_DIR = Path(".")
TABLE_DIR = OUTPUT_DIR / "results" / "tables"
FIGURE_DIR = OUTPUT_DIR / "results" / "figures"
for directory in (TABLE_DIR, FIGURE_DIR, Path("processors"), Path("data")):
    directory.mkdir(parents=True, exist_ok=True)

assert BASIS_GATES == ["rz", "sx", "x", "cx"]
assert OPT_LEVEL == 1 and SEED == SEED_TRANSPILER == SEED_SIMULATOR == 42
assert SHOTS == 4096
print(f"Python {platform.python_version()} | Qiskit official settings: "
      f"basis={BASIS_GATES}, optimization_level={OPT_LEVEL}, "
      f"seed_transpiler={SEED_TRANSPILER}, seed_simulator={SEED_SIMULATOR}, shots={SHOTS}")

Python 3.11.9 | Qiskit official settings: basis=['rz', 'sx', 'x', 'cx'], optimization_level=1, seed_transpiler=42, seed_simulator=42, shots=4096


## 1. Locked instance, QUBO, and exact classical baseline

For $x_i\in\{0,1\}$ (the group assignment of vertex $i$), the cut weight is

$$C(x)=\sum_{(i,j)\in E}w_{ij}(x_i+x_j-2x_ix_j),\qquad \max_x C(x).$$

We minimize $-C(x)$. The symmetric-matrix convention is $Q_{ii}=-\sum_{j:(i,j)\in E}w_{ij}$ and $Q_{ij}=Q_{ji}=w_{ij}$ for each edge; therefore $x^TQx=-C(x)$ with zero constant. With $x_i=(1-z_i)/2$, the positive-cut Hamiltonian is $H_C=\sum_{(i,j)}w_{ij}(I-Z_iZ_j)/2$ and we maximize $\langle H_C\rangle$; the mixer is $H_B=\sum_iX_i$.

In [2]:
with open("data/instance.json", encoding="utf-8") as instance_file:
    instance = json.load(instance_file)

n_vertices = instance["num_vertices"]
edges = [tuple(edge) for edge in instance["weighted_edges"]]
assert n_vertices == 6 and len(edges) == 9
assert sum(weight for _, _, weight in edges) == 27

graph = nx.Graph()
graph.add_nodes_from(range(n_vertices))
graph.add_weighted_edges_from(edges)

def cut_weight(bits):
    return sum(weight * (bits[u] + bits[v] - 2 * bits[u] * bits[v])
               for u, v, weight in edges)

qubo = np.zeros((n_vertices, n_vertices), dtype=int)
for u, v, weight in edges:
    qubo[u, u] -= weight
    qubo[v, v] -= weight
    qubo[u, v] += weight
    qubo[v, u] += weight

all_bits = list(itertools.product((0, 1), repeat=n_vertices))
cut_by_bits = {bits: cut_weight(bits) for bits in all_bits}
for bits in all_bits:
    x = np.asarray(bits, dtype=int)
    assert int(x @ qubo @ x) == -cut_weight(bits)  # constant term is zero

classical_optimum = max(cut_by_bits.values())
optimal_set = sorted(bits for bits, value in cut_by_bits.items()
                     if value == classical_optimum)
total_weight = sum(weight for _, _, weight in edges)
random_mean = sum(cut_by_bits.values()) / len(all_bits)
random_ratio = random_mean / classical_optimum
random_opt_probability = len(optimal_set) / len(all_bits)

expected_optimal_set = [(0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 0)]
assert classical_optimum == 22
assert optimal_set == expected_optimal_set
assert math.isclose(random_mean, 13.5)
assert math.isclose(random_ratio, 13.5 / 22)
assert math.isclose(random_opt_probability, 2 / 64)
assert all(tuple(1 - bit for bit in bits) in optimal_set for bits in optimal_set)

print("Loaded locked PS-O1 data from data/instance.json.")
print(f"Total edge weight={total_weight}; C*={classical_optimum}; "
      f"all optimal bitstrings={optimal_set}")
print(f"Exact uniform-random baseline: mean={random_mean:.3f}, "
      f"r_random={random_ratio:.6f} ({random_ratio:.3f}), "
      f"P_opt_random={random_opt_probability:.6f} ({random_opt_probability:.3f})")
print("QUBO assertion passed for all 64 assignments.")

Loaded locked PS-O1 data from data/instance.json.
Total edge weight=27; C*=22; all optimal bitstrings=[(0, 0, 1, 0, 1, 1), (1, 1, 0, 1, 0, 0)]
Exact uniform-random baseline: mean=13.500, r_random=0.613636 (0.614), P_opt_random=0.031250 (0.031)
QUBO assertion passed for all 64 assignments.


## 2. p=1 QAOA and cost-Hamiltonian sanity checks

For each edge, $e^{-i\gamma w(I-Z_iZ_j)/2}$ equals a global phase times $RZZ(-\gamma w)$. The `CX — RZ(-gamma*weight) — CX` implementation therefore has the requested sign and factor. The global phase does not affect measurement probabilities. The assertion below compares the explicit Hamiltonian expectation with the cut-function average over the statevector distribution.

In [3]:
def build_qaoa_circuit(gamma, beta, measure=False):
    circuit = QuantumCircuit(n_vertices, n_vertices if measure else 0)
    circuit.h(range(n_vertices))
    for u, v, weight in edges:
        circuit.cx(u, v)
        circuit.rz(-gamma * weight, v)
        circuit.cx(u, v)
    for qubit in range(n_vertices):
        circuit.rx(2 * beta, qubit)
    if measure:
        circuit.measure(range(n_vertices), range(n_vertices))
    return circuit

def cost_hamiltonian():
    terms = [("I" * n_vertices, total_weight / 2)]
    for u, v, weight in edges:
        pauli = ["I"] * n_vertices
        pauli[u] = "Z"
        pauli[v] = "Z"
        terms.append(("".join(reversed(pauli)), -weight / 2))
    return SparsePauliOp.from_list(terms)

H_C = cost_hamiltonian()

def expected_cut_statevector(gamma, beta):
    state = Statevector.from_instruction(build_qaoa_circuit(gamma, beta))
    probabilities = state.probabilities()
    enumerated = sum(
        probabilities[index] * cut_weight(tuple((index >> q) & 1 for q in range(n_vertices)))
        for index in range(2 ** n_vertices)
    )
    hamiltonian_value = float(np.real(state.expectation_value(H_C)))
    assert np.isclose(hamiltonian_value, enumerated, atol=1e-10)
    return enumerated

test_state = Statevector.from_instruction(build_qaoa_circuit(0.37, 0.29))
test_distribution_expectation = sum(
    test_state.probabilities()[index]
    * cut_weight(tuple((index >> q) & 1 for q in range(n_vertices)))
    for index in range(2 ** n_vertices)
)
assert np.isclose(float(np.real(test_state.expectation_value(H_C))),
                  test_distribution_expectation, atol=1e-10)
two_qubit_cost = QuantumCircuit(2)
two_qubit_cost.cx(0, 1)
two_qubit_cost.rz(-0.37 * 3, 1)
two_qubit_cost.cx(0, 1)
target_cost_diagonal = np.diag(np.exp(-1j * 0.37 * 3 * np.array([0, 1, 1, 0])))
assert np.allclose(
    Operator(two_qubit_cost).data,
    np.exp(1j * 0.37 * 3 / 2) * target_cost_diagonal,
    atol=1e-10,
)
assert counts_to_x("100101") == (1, 0, 1, 0, 0, 1)
assert counts_to_x("  100101 ") == (1, 0, 1, 0, 0, 1)
print("QAOA p=1 cost-angle and H_C expectation checks passed.")
print("Organizer count conversion check: '100101' ->", counts_to_x("100101"))

QAOA p=1 cost-angle and H_C expectation checks passed.
Organizer count conversion check: '100101' -> (1, 0, 1, 0, 0, 1)


## 3. Five seeded ideal optimizations

A shared 101 × 101 grid search covers $\gamma \in [0,2\pi]$ and $\beta \in [0,\pi]$. Its best point is locally polished, then compared with L-BFGS-B results from each of the five fixed seeded starts. The best exact statevector expected cut is retained per seed; the best row supplies the angles used for processor benchmarks.

In [ ]:
ranges = ((0.0, 2.0 * np.pi), (0.0, np.pi))
bounds = ((0.0, 2.0 * np.pi), (0.0, np.pi))


def negative_expected_cut(parameters):
    return -expected_cut_statevector(float(parameters[0]), float(parameters[1]))


grid_parameters, grid_objective, _, grid_values = brute(
    negative_expected_cut,
    ranges,
    Ns=GRID_POINTS,
    full_output=True,
    finish=None,
)
grid_evaluations = int(grid_values.size)

global_polish = minimize(
    negative_expected_cut,
    grid_parameters,
    method=OPTIMIZER_NAME,
    bounds=bounds,
    options={"maxiter": OPTIMIZER_MAXITER, "ftol": 1e-12},
)

seed_rows = []
for seed in INITIALIZATION_SEEDS:
    rng = np.random.default_rng(seed)
    initial_parameters = rng.uniform(
        [0.0, 0.0], [2.0 * np.pi, np.pi], size=2
    )
    seed_polish = minimize(
        negative_expected_cut,
        initial_parameters,
        method=OPTIMIZER_NAME,
        bounds=bounds,
        options={"maxiter": OPTIMIZER_MAXITER, "ftol": 1e-12},
    )

    candidates = [
        (np.asarray(grid_parameters), float(grid_objective), False)
    ]
    for result in (global_polish, seed_polish):
        if result.success and np.isfinite(result.fun):
            candidates.append(
                (np.asarray(result.x), float(result.fun), True)
            )

    best_parameters, best_objective, polish_success = min(
        candidates, key=lambda candidate: candidate[1]
    )
    expected_value = -best_objective
    seed_rows.append({
        "seed": seed,
        "initial_gamma": float(initial_parameters[0]),
        "initial_beta": float(initial_parameters[1]),
        "gamma": float(best_parameters[0]),
        "beta": float(best_parameters[1]),
        "expected_cut": expected_value,
        "r": expected_value / classical_optimum,
        "optimizer_success": bool(polish_success),
        "evaluations": int(
            grid_evaluations + global_polish.nfev + seed_polish.nfev
        ),
    })

seed_table = pd.DataFrame(seed_rows)
seed_mean_r = float(seed_table["r"].mean())
seed_std_r = float(seed_table["r"].std(ddof=0))
best_seed_row = seed_table.loc[seed_table["expected_cut"].idxmax()]
best_gamma = float(best_seed_row["gamma"])
best_beta = float(best_seed_row["beta"])
best_exact_cut = float(best_seed_row["expected_cut"])

display(seed_table)
print(
    f"Shared brute grid={GRID_POINTS}x{GRID_POINTS}; "
    f"local optimizer={OPTIMIZER_NAME}, maxiter={OPTIMIZER_MAXITER}."
)
print(
    f"Across starts: r mean={seed_mean_r:.6f}, "
    f"population std={seed_std_r:.6f}; best seed={int(best_seed_row['seed'])}, "
    f"gamma={best_gamma:.8f}, beta={best_beta:.8f}, "
    f"expected cut={best_exact_cut:.6f}."
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(seed_table["seed"].astype(str), seed_table["r"], color="#4C78A8")
ax.axhline(
    random_ratio,
    color="#F58518",
    linestyle="--",
    label=f"Random r={random_ratio:.3f}",
)
ax.set(
    title="Ideal statevector approximation ratio by initialization",
    xlabel="Initialization seed",
    ylabel="Expected cut / C*",
)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "seed_approximation_ratios.png", dpi=160)
plt.show()

,seed,initial_gamma,initial_beta,gamma,beta,expected_cut,r,optimizer_success,evaluations
0,0,2.001074,0.847560,1.355484,0.417955,15.493620,0.704255,True,41
1,1,1.607935,2.985970,1.575258,3.991206,15.532392,0.706018,True,52
2,2,0.821879,0.937738,0.813038,1.115452,15.557583,0.707163,True,33
3,3,0.269075,0.743962,1.355389,1.988618,15.493620,0.704255,True,41
4,4,2.962698,1.606383,2.960442,0.407144,15.551533,0.706888,True,48


COBYLA maxiter=200; exact statevector objective.
Across starts: r mean=0.705716, population std=0.001251; best seed=2, gamma=0.81303762, beta=1.11545202, expected cut=15.557583.


/var/folders/5h/186qflbn24dg0m_y9c4ln3z80000gn/T/ipykernel_8510/199846316.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. A/B bottleneck first; freeze and validate Processor D

The official metrics are computed on the bound p=1 circuit with final measurements. Processor A is expected to be non-executable at six logical qubits. First measure A and B; then load the frozen D JSON and validate it. Processor D embeds the locked graph interactions directly, but zero routing is treated as a measured result, not an assumption.

In [5]:
bound_qaoa = build_qaoa_circuit(best_gamma, best_beta, measure=True)
metrics_a = circuit_metrics(bound_qaoa, PROCESSOR_A)
metrics_b = circuit_metrics(bound_qaoa, PROCESSOR_B)
print("Measured Processor A bottleneck:", metrics_a)
print("Measured Processor B bottleneck:", metrics_b)
assert metrics_a["executable"] is False
assert "needs 6 qubits" in metrics_a["reason"]

processor_d = load_processor("processors/processor_D.json")
d_errors = validate_processor_d(processor_d, required_qubits=n_vertices)
assert d_errors == [], f"Processor D validation errors: {d_errors}"
d_sha256 = hashlib.sha256(Path("processors/processor_D.json").read_bytes()).hexdigest()
metrics_d = circuit_metrics(bound_qaoa, processor_d)
assert metrics_d["executable"] is True
print("Processor D rule check: OK")
print("Frozen Processor D SHA-256:", d_sha256)
print("Measured Processor D bottleneck:", metrics_d)
print(f"B -> D measured changes: swaps {metrics_b['swaps']} -> {metrics_d['swaps']}, "
      f"CX overhead {metrics_b['cx_overhead']} -> {metrics_d['cx_overhead']}, "
      f"depth {metrics_b['depth']} -> {metrics_d['depth']}.")

Measured Processor A bottleneck: {'processor': 'Processor A', 'executable': False, 'reason': 'needs 6 qubits > 5'}
Measured Processor B bottleneck: {'processor': 'Processor B', 'executable': True, 'logical_qubits': 6, 'physical_qubits': 9, 'logical_cx': 18, 'cx': 21, 'one_q_gates': 57, 'depth': 37, 'two_q_depth': 17, 'swaps': 1, 'cx_overhead': 3}
Processor D rule check: OK
Frozen Processor D SHA-256: 8b88e1f5ced09531fe674cc48731162fdbf7db72b915bf4e44be5e608116a1e8
Measured Processor D bottleneck: {'processor': 'Processor D', 'executable': True, 'logical_qubits': 6, 'physical_qubits': 6, 'logical_cx': 18, 'cx': 18, 'one_q_gates': 57, 'depth': 29, 'two_q_depth': 14, 'swaps': 0, 'cx_overhead': 0}
B -> D measured changes: swaps 1 -> 0, CX overhead 3 -> 0, depth 37 -> 29.


## 5. Required ideal/noisy processor benchmarks

The same optimized angles are used for each executable processor and both noise conditions. Transpilation and simulation use only the organizer helpers. For Processor A, both condition rows are retained and explicitly marked non-executable; the graph is not reduced or remapped to five qubits.

In [6]:
processors = {"A": PROCESSOR_A, "B": PROCESSOR_B, "D": processor_d}
processor_metrics = {"A": metrics_a, "B": metrics_b, "D": metrics_d}
quality_rows = []
result_rows = []
all_counts = {}
transpiled_circuits = {}

for processor_key, processor in processors.items():
    metrics = processor_metrics[processor_key]
    if processor_key != "A":
        transpiled_circuits[processor_key] = transpile_for(bound_qaoa, processor)
        assert transpiled_circuits[processor_key].num_clbits == n_vertices
    for condition in ("ideal", "noisy"):
        if not metrics["executable"]:
            result_rows.append({
                "problem_id": "PS-O1", "processor": processor_key, "condition": condition,
                "executable": "false", "logical_qubits": n_vertices,
                "physical_qubits": processor["num_qubits"], "logical_cx": "",
                "cx": "", "depth": "", "two_q_depth": "", "swaps": "",
                "cx_overhead": "", "shots": "", "quality_metric": "",
                "quality_value": "", "p_opt": "", "classical_optimum": classical_optimum,
                "random_baseline": round(random_ratio, 3),
            })
            continue

        executable_circuit = transpiled_circuits[processor_key]
        backend = simulator(noisy=(condition == "noisy"), processor=processor)
        counts = backend.run(executable_circuit, shots=SHOTS).result().get_counts()
        assert sum(counts.values()) == SHOTS
        all_counts[(processor_key, condition)] = counts
        cut_total = 0
        optimal_shots = 0
        best_cut = -1
        for key, count in counts.items():
            assignment = counts_to_x(key)
            assert len(assignment) == n_vertices, (key, assignment)
            value = cut_weight(assignment)
            cut_total += value * count
            best_cut = max(best_cut, value)
            if assignment in optimal_set:
                optimal_shots += count
        mean_cut = cut_total / SHOTS
        approximation_ratio = mean_cut / classical_optimum
        best_ratio = best_cut / classical_optimum
        optimal_probability = optimal_shots / SHOTS
        quality_rows.append({
            "processor": processor_key, "condition": condition,
            "best_cut": best_cut, "r": approximation_ratio, "r_best": best_ratio,
            "P_opt": optimal_probability, "random_r": random_ratio,
            "random_P_opt": random_opt_probability, "mean_cut": mean_cut,
            "shots": SHOTS,
        })
        result_rows.append({
            "problem_id": "PS-O1", "processor": processor_key, "condition": condition,
            "executable": "true", "logical_qubits": metrics["logical_qubits"],
            "physical_qubits": metrics["physical_qubits"],
            "logical_cx": metrics["logical_cx"], "cx": metrics["cx"],
            "depth": metrics["depth"], "two_q_depth": metrics["two_q_depth"],
            "swaps": metrics["swaps"], "cx_overhead": metrics["cx_overhead"],
            "shots": SHOTS, "quality_metric": "approx_ratio",
            "quality_value": approximation_ratio, "p_opt": optimal_probability,
            "classical_optimum": classical_optimum,
            "random_baseline": round(random_ratio, 3),
        })
        print(f"{processor_key} {condition}: best={best_cut}, r={approximation_ratio:.6f}, "
              f"r_best={best_ratio:.6f}, P_opt={optimal_probability:.6f}; "
              f"random r={random_ratio:.6f}, random P_opt={random_opt_probability:.6f}")

quality_table = pd.DataFrame(quality_rows)
results_table = pd.DataFrame(result_rows)
resource_rows = []
for processor_key in ("A", "B", "D"):
    row = {**processor_metrics[processor_key], "processor": processor_key}
    resource_rows.append(row)
resource_table = pd.DataFrame(resource_rows)
noise_gap_rows = []
for processor_key in ("B", "D"):
    ideal_row = quality_table[quality_table["processor"] == processor_key].iloc[0]
    noisy_row = quality_table[quality_table["processor"] == processor_key].iloc[1]
    noise_gap_rows.append({
        "processor": processor_key,
        "delta_r_noisy_minus_ideal": noisy_row["r"] - ideal_row["r"],
        "delta_r_best_noisy_minus_ideal": noisy_row["r_best"] - ideal_row["r_best"],
        "delta_P_opt_noisy_minus_ideal": noisy_row["P_opt"] - ideal_row["P_opt"],
    })
noise_gap_table = pd.DataFrame(noise_gap_rows)
display(quality_table)
display(resource_table)
display(noise_gap_table)
results_table = results_table[[
    "problem_id", "processor", "condition", "executable", "logical_qubits",
    "physical_qubits", "logical_cx", "cx", "depth", "two_q_depth", "swaps",
    "cx_overhead", "shots", "quality_metric", "quality_value", "p_opt",
    "classical_optimum", "random_baseline",
]]
results_table.to_csv(TABLE_DIR / "results.csv", index=False)
quality_table.to_csv(TABLE_DIR / "quality.csv", index=False)
resource_table.to_csv(TABLE_DIR / "resources.csv", index=False)
noise_gap_table.to_csv(TABLE_DIR / "noise_gaps.csv", index=False)
assert len(results_table) == 6
print(f"Saved official-format result rows to {TABLE_DIR / 'results.csv'}.")

B ideal: best=22, r=0.711847, r_best=1.000000, P_opt=0.072998; random r=0.613636, random P_opt=0.031250
B noisy: best=22, r=0.701483, r_best=1.000000, P_opt=0.069824; random r=0.613636, random P_opt=0.031250
D ideal: best=22, r=0.706610, r_best=1.000000, P_opt=0.069580; random r=0.613636, random P_opt=0.031250
D noisy: best=22, r=0.697521, r_best=1.000000, P_opt=0.064209; random r=0.613636, random P_opt=0.031250


,processor,condition,best_cut,r,r_best,P_opt,random_r,random_P_opt,mean_cut,shots
0,B,ideal,22,0.711847,1.0,0.072998,0.613636,0.03125,15.660645,4096
1,B,noisy,22,0.701483,1.0,0.069824,0.613636,0.03125,15.432617,4096
2,D,ideal,22,0.706610,1.0,0.069580,0.613636,0.03125,15.545410,4096
3,D,noisy,22,0.697521,1.0,0.064209,0.613636,0.03125,15.345459,4096


,processor,executable,reason,logical_qubits,physical_qubits,logical_cx,cx,one_q_gates,depth,two_q_depth,swaps,cx_overhead
0,A,False,needs 6 qubits > 5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B,True,NaN,6.0,9.0,18.0,21.0,57.0,37.0,17.0,1.0,3.0
2,D,True,NaN,6.0,6.0,18.0,18.0,57.0,29.0,14.0,0.0,0.0


,processor,delta_r_noisy_minus_ideal,delta_r_best_noisy_minus_ideal,delta_P_opt_noisy_minus_ideal
0,B,-0.010365,0.0,-0.003174
1,D,-0.009089,0.0,-0.005371


Saved official-format result rows to results/tables/results.csv.


### Resource efficiency: Processor D versus B at equal ideal quality

The exact statevector ideal quality is common to the same logical circuit; hardware compilation changes routing, not the intended unitary. The table compares measured resource counts and ideal sampled quality. Because finite-shot values may differ slightly, the exact statevector ratio is the equal-quality reference.

In [7]:
exact_ideal_r = best_exact_cut / classical_optimum
b_metrics = metrics_b
d_metrics = metrics_d
cx_savings = 100 * (1 - d_metrics["cx"] / b_metrics["cx"])
depth_savings = 100 * (1 - d_metrics["depth"] / b_metrics["depth"])
two_q_savings = 100 * (1 - d_metrics["two_q_depth"] / b_metrics["two_q_depth"])
efficiency_table = pd.DataFrame([{
    "comparison": "D vs B (same logical circuit; exact ideal r)",
    "exact_ideal_r_both": exact_ideal_r,
    "B_CX": b_metrics["cx"], "D_CX": d_metrics["cx"],
    "CX_reduction_percent": cx_savings,
    "B_depth": b_metrics["depth"], "D_depth": d_metrics["depth"],
    "depth_reduction_percent": depth_savings,
    "B_two_q_depth": b_metrics["two_q_depth"],
    "D_two_q_depth": d_metrics["two_q_depth"],
    "two_q_depth_reduction_percent": two_q_savings,
}])
display(efficiency_table)
print("Processor A capacity rule: not executable: needs 6 qubits, has 5. "
      "Its missing qubit prevents standard execution of the unchanged six-variable instance; "
      "no vertex is removed or compressed.")

,comparison,exact_ideal_r_both,B_CX,D_CX,CX_reduction_percent,B_depth,D_depth,depth_reduction_percent,B_two_q_depth,D_two_q_depth,two_q_depth_reduction_percent
0,D vs B (same logical circuit; exact ideal r),0.707163,21,18,14.285714,37,29,21.621622,17,14,17.647059


Processor A capacity rule: not executable: needs 6 qubits, has 5. Its missing qubit prevents standard execution of the unchanged six-variable instance; no vertex is removed or compressed.


## 6. Bell-state verification on Processor D

The organizer `bell_experiment` runs XX, YY, and ZZ for both selected physical pairs under ideal and official noisy simulation (12 basis-specific circuits total). The adjacent pair is a physical D edge; the farthest pair is selected by maximum graph distance. Fidelity is computed from all three correlators, never from computational-basis populations.

In [8]:
d_graph = nx.Graph()
d_graph.add_nodes_from(range(processor_d["num_qubits"]))
d_graph.add_edges_from(processor_d["edges"])
diameter = nx.diameter(d_graph)
adjacent_pair = tuple(processor_d["edges"][0])
farthest_pair = next(
    (u, v) for u, v in itertools.combinations(range(processor_d["num_qubits"]), 2)
    if nx.shortest_path_length(d_graph, u, v) == diameter
)
bell_rows = []
for pair_name, pair in (("adjacent", adjacent_pair), ("farthest", farthest_pair)):
    for noisy in (False, True):
        values = bell_experiment(processor_d, pair, noisy=noisy, shots=SHOTS)
        bell_rows.append({
            "pair": f"{pair[0]}-{pair[1]} ({pair_name})",
            "pair_distance": values["pair_distance"],
            "cx": values["cx"],
            "XX": values["XX"], "XX_se": values["XX_se"],
            "YY": values["YY"], "YY_se": values["YY_se"],
            "ZZ": values["ZZ"], "ZZ_se": values["ZZ_se"],
            "F": values["fidelity"], "F_se": values["fidelity_se"],
            "condition": "noisy" if noisy else "ideal",
        })
bell_table = pd.DataFrame(bell_rows, columns=[
    "pair", "pair_distance", "cx", "XX", "XX_se", "YY", "YY_se",
    "ZZ", "ZZ_se", "F", "F_se", "condition",
])
assert len(bell_table) == 4  # each row contains all three bases; 12 circuits total
assert bell_table["condition"].value_counts().to_dict() == {"ideal": 2, "noisy": 2}
bell_table.to_csv(TABLE_DIR / "bell.csv", index=False)
display(bell_table)
print(f"Processor D diameter={diameter}; adjacent pair={adjacent_pair}; "
      f"farthest pair={farthest_pair}; 3 bases x 2 pairs x 2 conditions = 12 circuits.")
print("Fidelity uses F=(1+XX-YY+ZZ)/4 and SE(F)=sqrt(SE_XX^2+SE_YY^2+SE_ZZ^2)/4.")

,pair,pair_distance,cx,XX,XX_se,YY,YY_se,ZZ,ZZ_se,F,F_se,condition
0,0-2 (adjacent),1,1,1.000000,0.000000,-1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,ideal
1,0-2 (adjacent),1,1,0.945801,0.005074,-0.952148,0.004776,0.955078,0.004631,0.963257,0.002092,noisy
2,0-1 (farthest),2,4,1.000000,0.000000,-1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,ideal
3,0-1 (farthest),2,4,0.944336,0.005140,-0.934082,0.005579,0.938477,0.005396,0.954224,0.002327,noisy


Processor D diameter=2; adjacent pair=(0, 2); farthest pair=(0, 1); 3 bases x 2 pairs x 2 conditions = 12 circuits.
Fidelity uses F=(1+XX-YY+ZZ)/4 and SE(F)=sqrt(SE_XX^2+SE_YY^2+SE_ZZ^2)/4.


## 7. Figures: cut distribution, partition, resources, and topologies

The selected partition is the best sampled cut from Processor D noisy results. The plots and graph drawings are saved under `results/figures/` and are regenerated on every run.

In [9]:
d_noisy_counts = all_counts[("D", "noisy")]
sampled_values = []
sampled_assignments = []
for key, count in d_noisy_counts.items():
    assignment = counts_to_x(key)
    sampled_values.extend([cut_weight(assignment)] * count)
    sampled_assignments.extend([assignment] * count)
best_sampled_cut = max(sampled_values)
best_sampled_assignments = sorted({
    bits for bits, value in zip(sampled_assignments, sampled_values)
    if value == best_sampled_cut
})
best_partition = best_sampled_assignments[0]

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.arange(-0.5, total_weight + 1.5, 1)
ax.hist(sampled_values, bins=bins, color="#4C78A8", edgecolor="white")
ax.axvline(classical_optimum, color="#E45756", linestyle="--",
           label=f"Optimal C*={classical_optimum}; bitstrings={optimal_set}")
ax.set(title="Processor D noisy QAOA: sampled cut-value histogram",
       xlabel="Cut weight", ylabel="Shots")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "cut_value_histogram.png", dpi=160)
plt.show()

positions = nx.spring_layout(graph, seed=42)
node_colors = ["#4C78A8" if best_partition[i] == 0 else "#F58518"
               for i in range(n_vertices)]
cut_edges = [(u, v) for u, v, _ in edges if best_partition[u] != best_partition[v]]
within_edges = [(u, v) for u, v, _ in edges if best_partition[u] == best_partition[v]]
fig, ax = plt.subplots(figsize=(7, 6))
nx.draw_networkx_nodes(graph, positions, node_color=node_colors, node_size=800, ax=ax)
nx.draw_networkx_labels(graph, positions, font_color="white", font_weight="bold", ax=ax)
nx.draw_networkx_edges(graph, positions, edgelist=within_edges, edge_color="#999999",
                       width=1.5, ax=ax)
nx.draw_networkx_edges(graph, positions, edgelist=cut_edges, edge_color="#E45756",
                       width=3, ax=ax)
nx.draw_networkx_edge_labels(graph, positions,
                             edge_labels=nx.get_edge_attributes(graph, "weight"), ax=ax)
ax.set_title(f"Best Processor D noisy sample {best_partition}; red edges are cut")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "best_partition.png", dpi=160)
plt.show()

for key, processor in processors.items():
    coupling = nx.Graph()
    coupling.add_nodes_from(range(processor["num_qubits"]))
    coupling.add_edges_from(processor["edges"])
    pos = nx.spring_layout(coupling, seed=42)
    fig, ax = plt.subplots(figsize=(6, 5))
    nx.draw_networkx(coupling, pos, node_color="#72B7B2", node_size=650,
                     with_labels=True, ax=ax)
    ax.set_title(f"{processor['name']} coupling graph")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"processor_{key}_coupling.png", dpi=160)
    plt.show()

executable_resources = resource_table[resource_table["processor"].isin(["B", "D"])]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric_name in zip(axes, ("cx", "depth", "two_q_depth")):
    values = executable_resources.set_index("processor")[metric_name]
    values.plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518"])
    ax.set_title(metric_name.replace("_", " "))
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
fig.suptitle("Official QAOA resource metrics (A is not executable)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "resource_comparison.png", dpi=160)
plt.show()
print(f"Saved figures to {FIGURE_DIR.resolve()}; best sampled cut={best_sampled_cut}, "
      f"partition={best_partition}.")

/var/folders/5h/186qflbn24dg0m_y9c4ln3z80000gn/T/ipykernel_8510/4146436651.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved figures to /Users/adabalaramakrishna/qff-2026-quantum-drift-PS-O1/Phase 2/results/figures; best sampled cut=22, partition=(0, 0, 1, 0, 1, 1).


/var/folders/5h/186qflbn24dg0m_y9c4ln3z80000gn/T/ipykernel_8510/4146436651.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5h/186qflbn24dg0m_y9c4ln3z80000gn/T/ipykernel_8510/4146436651.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/5h/186qflbn24dg0m_y9c4ln3z80000gn/T/ipykernel_8510/4146436651.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Evidence-based analysis and report generation

Processor A's five-qubit capacity is insufficient for the six-vertex instance, so it has no ideal/noisy sample metrics. Processor B requires routing for this optimized p=1 circuit; Processor D is an application-specific embedding. The official noise model applies 0.5% two-qubit depolarizing error per CX and 1% symmetric readout bit-flip; extra routed gates increase exposure, while real hardware can also have crosstalk and calibration variation absent from this model.

The final report and README below are generated from the executed result tables. They do not contain hand-entered performance values.

In [10]:
quality_lookup = quality_table.set_index(["processor", "condition"])
bell_lookup = bell_table.set_index(["pair", "condition"])

def metric_value(processor_key, name):
    value = processor_metrics[processor_key].get(name, np.nan)
    return "N/A" if pd.isna(value) else str(int(value))

result_markdown_rows = []
for _, row in results_table.iterrows():
    if row["executable"] == "true":
        q = quality_lookup.loc[(row["processor"], row["condition"])]
        result_markdown_rows.append(
            f"| {row['processor']} | {row['condition']} | yes | {q['best_cut']} | "
            f"{q['r']:.4f} | {q['r_best']:.4f} | {q['P_opt']:.4f} | "
            f"{random_ratio:.3f} / {random_opt_probability:.3f} |"
        )
    else:
        result_markdown_rows.append(
            f"| A | {row['condition']} | no (6 needed; 5 available) | — | — | — | — | "
            f"{random_ratio:.3f} / {random_opt_probability:.3f} |"
        )
result_markdown = "\n".join(result_markdown_rows)
seed_markdown = "\n".join(
    f"| {int(row.seed)} | {row.expected_cut:.6f} | {row.r:.6f} |"
    for row in seed_table.itertuples()
)
resource_markdown = "\n".join(
    f"| {key} | {metric_value(key, 'logical_qubits')} | "
    f"{metric_value(key, 'physical_qubits')} | {metric_value(key, 'logical_cx')} | "
    f"{metric_value(key, 'cx')} | {metric_value(key, 'one_q_gates')} | "
    f"{metric_value(key, 'depth')} | {metric_value(key, 'two_q_depth')} | "
    f"{metric_value(key, 'swaps')} | {metric_value(key, 'cx_overhead')} |"
    for key in ("A", "B", "D")
)
bell_markdown = "\n".join(
    f"| {row.pair} | {row.pair_distance} | {row.condition} | "
    f"{row.XX:.4f} ± {row.XX_se:.4f} | {row.YY:.4f} ± {row.YY_se:.4f} | "
    f"{row.ZZ:.4f} ± {row.ZZ_se:.4f} | {row.F:.4f} ± {row.F_se:.4f} |"
    for row in bell_table.itertuples()
)
b_noisy = quality_lookup.loc[("B", "noisy")]
d_noisy = quality_lookup.loc[("D", "noisy")]
b_ideal = quality_lookup.loc[("B", "ideal")]
d_ideal = quality_lookup.loc[("D", "ideal")]
noise_gap_lookup = noise_gap_table.set_index("processor")
b_noise_gap = noise_gap_lookup.loc["B"]
d_noise_gap = noise_gap_lookup.loc["D"]
adj_ideal = bell_table[(bell_table["pair"].str.contains("adjacent")) &
                       (bell_table["condition"] == "ideal")].iloc[0]
adj_noisy = bell_table[(bell_table["pair"].str.contains("adjacent")) &
                       (bell_table["condition"] == "noisy")].iloc[0]
far_ideal = bell_table[(bell_table["pair"].str.contains("farthest")) &
                       (bell_table["condition"] == "ideal")].iloc[0]
far_noisy = bell_table[(bell_table["pair"].str.contains("farthest")) &
                       (bell_table["condition"] == "noisy")].iloc[0]

report = f'''# QFF 2026 Phase 2 Final Report — PS-O1

**Team:** Quantum Drifters  
**Problem:** Quantum Optimization of Network Partitioning Using QAOA  
**Locked instance:** `data/instance.json`  
**Frozen Processor D SHA-256:** `{d_sha256}`

## 1. Problem and formulation

The six-vertex weighted graph is loaded from the PS-O1 data file. For binary group assignments $x_i\\in\\{{0,1\\}}$, the objective is

$$C(x)=\\sum_{{(i,j)\\in E}} w_{{ij}}(x_i+x_j-2x_ix_j),\\qquad \\max_x C(x).$$

The QUBO minimized by brute-force verification is $-C(x)$. Its symmetric matrix uses $Q_{{ii}}=-\\sum_j w_{{ij}}$ and $Q_{{ij}}=Q_{{ji}}=w_{{ij}}$, with zero constant, and was asserted equal to $-C(x)$ for every one of the 64 assignments. With $x_i=(1-z_i)/2$, the positive-cut Hamiltonian is $H_C=\\sum w_{{ij}}(I-Z_iZ_j)/2$ and is maximized; $H_B=\\sum_i X_i$. The p=1 cost implementation uses `CX-RZ(-gamma*w)-CX`, which equals the required cost evolution up to a global phase.

Exact enumeration found total edge weight {total_weight}, classical optimum $C^*={classical_optimum}$, and the complete optimal set `{optimal_set}`. Uniform random enumeration gives mean {random_mean:.3f}, $r_{{random}}={random_ratio:.6f}$ ({random_ratio:.3f}), and $P_{{opt,random}}={random_opt_probability:.6f}$ ({random_opt_probability:.3f}).

The p=1 angles were optimized once per initial seed using {OPTIMIZER_NAME}, maximum {OPTIMIZER_MAXITER} iterations, and exact statevector expected cut. This avoids shot noise during optimization while keeping the final hardware comparison sampled:

| Seed | Exact expected cut | $r$ |
|---:|---:|---:|
{seed_markdown}

Across seeds, mean $r={seed_mean_r:.6f}$ and population standard deviation $={seed_std_r:.6f}$. The best optimized seed was {int(best_seed_row['seed'])}; the same resulting angles were used in every processor evaluation.

## 2. Bell-state results and interpretation

The organizer Bell routine measured XX, YY, and ZZ for an adjacent physical pair `{adjacent_pair}` (distance {int(adj_ideal.pair_distance)}) and farthest pair `{farthest_pair}` (distance {diameter}), each under ideal and official noisy conditions, for 4096 shots per basis. Fidelity and its standard error use $F=(1+\\langle XX\\rangle-\\langle YY\\rangle+\\langle ZZ\\rangle)/4$ and the quadrature formula:

| Pair | Distance | Condition | XX | YY | ZZ | Fidelity |
|---|---:|---|---:|---:|---:|---:|
{bell_markdown}

Ideal fidelity is {adj_ideal.F:.4f} for the adjacent pair and {far_ideal.F:.4f} for the farthest pair. The measured noisy fidelities are {adj_noisy.F:.4f} and {far_noisy.F:.4f}, respectively. The noisy gap reflects the organizer depolarizing and readout channels; routing a non-adjacent CX can add entangling operations and their associated error exposure. In this D graph, the distant pair's measured extra CX count is {int(far_noisy.cx) - int(adj_noisy.cx)} relative to the adjacent pair in the ZZ circuits. The official simulator applies 0.5% two-qubit depolarizing error per CX, 0.1% one-qubit depolarizing error, and 1% symmetric readout bit-flip. Readout affects both pairs, while routing can compound two-qubit error exposure for the distant pair. The synthetic model has no T1/T2 drift or crosstalk, so hardware effects beyond the specified channels are not represented; the report does not assume the distant fidelity must be worse unless the measured values show it.

## 3. Processor D design and measured justification

Before designing D, the optimized circuit was measured on the references: Processor A has {PROCESSOR_A['num_qubits']} physical qubits and cannot execute the six-qubit circuit. Processor B measured {metrics_b['swaps']} routed SWAP, {metrics_b['cx_overhead']} CX overhead, and depth {metrics_b['depth']}. D freezes a six-qubit topology equal to the nine locked interaction edges, with maximum degree four and the official noise rates. The organizer validator returned an empty error list.

Measured D resources are {metrics_d['swaps']} SWAP, {metrics_d['cx_overhead']} CX overhead, and depth {metrics_d['depth']}; thus the intended routing reduction was verified. The exact ideal statevector quality is identical for B and D at $r={exact_ideal_r:.6f}$. At this equal ideal quality, D uses {cx_savings:.2f}% fewer transpiled CX, {depth_savings:.2f}% less circuit depth, and {two_q_savings:.2f}% less two-qubit depth than B. The design is intentionally specialized to PS-O1; its degree-four connectivity may be less useful for other workloads and real hardware may face additional crosstalk.

## 4. Results on A/B/D, ideal/noisy, against baselines

The primary ratio is the sample mean cut divided by $C^*$; $r_{{best}}$ is the largest sampled cut divided by $C^*$; $P_{{opt}}$ counts either optimal complementary assignment. Every executable condition uses 4096 shots and the same optimized parameters.

| Processor | Condition | Executable | Best cut | $r$ | $r_{{best}}$ | $P_{{opt}}$ | Random $r$ / $P_{{opt}}$ |
|---|---|---:|---:|---:|---:|---:|---:|
{result_markdown}

Processor A is reported as `not executable: needs 6 qubits, has 5`; the locked graph remains unchanged. Its two required condition rows are present but no results are fabricated. B noisy measured $r={b_noisy.r:.6f}$ and $P_{{opt}}={b_noisy.P_opt:.6f}$; D noisy measured $r={d_noisy.r:.6f}$ and $P_{{opt}}={d_noisy.P_opt:.6f}$. Ideal finite-shot ratios were B {b_ideal.r:.6f} and D {d_ideal.r:.6f}. Noise-related changes (noisy minus ideal) are B $\Delta r={b_noise_gap.delta_r_noisy_minus_ideal:+.6f}$ and $\Delta P_{{opt}}={b_noise_gap.delta_P_opt_noisy_minus_ideal:+.6f}$; D $\Delta r={d_noise_gap.delta_r_noisy_minus_ideal:+.6f}$ and $\Delta P_{{opt}}={d_noise_gap.delta_P_opt_noisy_minus_ideal:+.6f}$. All values are in `results/tables/results.csv`, `results/tables/quality.csv`, and `results/tables/noise_gaps.csv`.

Official resource metrics from `circuit_metrics` on the bound circuit with measurements:

| Processor | Logical qubits | Physical qubits | Logical CX | CX | One-qubit gates | Depth | Two-qubit depth | SWAPs | CX overhead |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
{resource_markdown}

The A row is non-executable because the processor has one fewer qubit than the unmodified problem. B routing increases CX/depth relative to D; the noisy quality difference is measured in the benchmark table rather than inferred from gate counts alone. Relative to the random baseline, the QAOA noisy results are {b_noisy.r - random_ratio:+.6f} in $r$ for B and {d_noisy.r - random_ratio:+.6f} for D.

### Results figures

![Weighted instance and best sampled partition](results/figures/best_partition.png)

![Cut-value distribution](results/figures/cut_value_histogram.png)

![Seed sensitivity](results/figures/seed_approximation_ratios.png)

![Resource comparison](results/figures/resource_comparison.png)

Processor coupling maps: [A](results/figures/processor_A_coupling.png), [B](results/figures/processor_B_coupling.png), and [D](results/figures/processor_D_coupling.png).

## 5. Trade-offs, limitations, and next improvements

The circuit is limited to p=1 and only five initializations; the observed across-seed standard deviation is {seed_std_r:.6f}, which quantifies optimizer sensitivity for these starts but not all possible optimization error. Ideal/noisy sampling has finite-shot uncertainty at 4096 shots. The local Aer model is synthetic and omits calibration drift, crosstalk, leakage, and pulse scheduling. Processor D is tailored to the PS-O1 interaction graph, so its benefits should not be generalized to unrelated graphs.

Evidence-based next steps are to repeat the five-start optimizer with a larger preregistered seed set, compare p=2 as a separately labeled experiment under the same runtime budget, and test topology robustness on additional locked graph instances. The observed B and D noise-related changes in $r$ are {b_noise_gap.delta_r_noisy_minus_ideal:+.6f} and {d_noise_gap.delta_r_noisy_minus_ideal:+.6f}; mitigation work should be prioritized based on these measured gaps rather than assumed benefits. Any physical-hardware conclusions require calibration data; this submission reports only the organizer-defined simulator results.

## Reproducibility

The notebook loads the graph from `data/instance.json`, imports the unmodified organizer utilities, records all seeds, validates the frozen D JSON, and regenerates tables and figures. The exact Processor D file hash is `{d_sha256}`. Run `python -m venv opt-env`, activate it, install `requirements.txt`, then execute `jupyter nbconvert --execute --to notebook main.ipynb`. The explicit output format is required by current nbconvert releases.
'''
Path("report.md").write_text(report, encoding="utf-8")

readme_rows = []
for _, result_row in results_table.iterrows():
    if result_row["executable"] == "true":
        quality_row = quality_lookup.loc[
            (result_row["processor"], result_row["condition"])
        ]
        readme_rows.append(
            f"| {result_row['processor']} | {result_row['condition']} | yes | "
            f"{quality_row['best_cut']} | {quality_row['r']:.4f} | "
            f"{quality_row['r_best']:.4f} | {quality_row['P_opt']:.4f} |"
        )
    else:
        readme_rows.append(
            f"| A | {result_row['condition']} | no (6 needed; 5 available) | "
            "— | — | — | — |"
        )
readme_table_rows = "\n".join(readme_rows)
readme = f'''# QFF 2026 Phase 2 — PS-O1

**Team:** Quantum Drifters  
**Problem statement:** PS-O1, weighted six-vertex Max-Cut  
**Processor D SHA-256:** `{d_sha256}`

## Overview

`main.ipynb` loads the locked graph from `data/instance.json`, verifies the QUBO and exact optimum, optimizes p=1 QAOA with five fixed starts, evaluates Processor B and the frozen Processor D in ideal/noisy simulation, records Processor A's capacity limit, runs the required Bell checks, and regenerates result tables and figures. The official organizer module `qff_phase2.py` is unchanged.

## Results (4096 shots; random baseline r={random_ratio:.3f}, P_opt={random_opt_probability:.3f})

| Processor | Condition | Executable | Best cut | r | r_best | P_opt |
|---|---|---:|---:|---:|---:|---:|
{readme_table_rows}

See `report.md`, `results/tables/`, and `results/figures/` for full analysis and artifacts.

## Reproduce

Python 3.10–3.12:

```bash
python -m venv opt-env
source opt-env/bin/activate
pip install -r requirements.txt
jupyter nbconvert --execute --to notebook main.ipynb
```

On Windows, activate with `opt-env\\Scripts\\activate`.

## AI-use disclosure

GitHub Copilot (AI assistant using Copilot SDK in VS Code) was used to help draft and validate notebook code, visualizations, and documentation. The team is responsible for checking the locked inputs, organizer utilities, and all reported results.
'''
Path("README.md").write_text(readme, encoding="utf-8")
print("Generated report.md and README.md from executed benchmark values.")
print(f"Noisy B: r={b_noisy.r:.6f}, P_opt={b_noisy.P_opt:.6f}; "
      f"Noisy D: r={d_noisy.r:.6f}, P_opt={d_noisy.P_opt:.6f}.")
print(f"Bell noisy adjacent F={adj_noisy.F:.6f}; noisy farthest F={far_noisy.F:.6f}.")

Generated report.md and README.md from executed benchmark values.
Noisy B: r=0.701483, P_opt=0.069824; Noisy D: r=0.697521, P_opt=0.064209.
Bell noisy adjacent F=0.963257; noisy farthest F=0.954224.


## 9. Interpretation and conclusion

Processor A's capacity limit makes the required circuit non-executable without changing the locked problem, so its ideal/noisy results remain intentionally blank. Processor B's routing overhead and Processor D's measured direct embedding quantify the connectivity trade-off for this particular circuit. Compare noisy ratios and optimal probabilities against the exact random baseline before drawing conclusions; the custom D topology is a targeted design, not a general-purpose hardware recommendation.

### Rubric checklist

- **C1 Bell verification:** `bell_experiment` outputs and `results/tables/bell.csv`; adjacent and maximum-distance D pairs, ideal/noisy, XX/YY/ZZ, standard errors, and fidelity.
- **C2 Processor D:** `processors/processor_D.json`, validation assertion, SHA-256, A/B/D official metrics, coupling drawing, and measured bottleneck comparison.
- **C3 formulation and benchmarking:** QUBO assertion over 64 states, complete exact optimum, p=1 QAOA, five seeded runs, ideal/noisy quality table, and official resource metrics.
- **C4 reproducibility:** pinned `requirements.txt`, fixed official seeds/settings, `README.md` commands, and this top-to-bottom notebook.
- **C5 analysis and figures:** seed plot, cut histogram, best partition, resource comparison, and A/B/D topology figures under `results/figures/`.